## IMPORTING LIBRARIES

In [1]:
import re, json, warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")

## NLP PREPROCESSING

In [2]:
STOPWORDS = {
    "a","an","the","and","but","or","so","yet","for","as","at","by","from","in",
    "into","of","off","on","onto","out","over","to","up","via","with","i","me",
    "my","we","our","you","your","he","him","his","she","her","they","them",
    "their","this","that","these","those","it","its","is","am","are","was",
    "were","be","been","being","have","has","had","do","does","did","will",
    "would","shall","should","may","might","must","can","could","also","just",
    "very","quite","more","most","much","many","some","any","all","each","every",
    "than","then","there","here","how","when","where","who","which","what","why",
    "about","above","after","again","against","because","before","between","during",
    "if","only","own","same","through","too","under","until","while","within",
    "ll","ve","re","d","m","s","t"
}

NEGATIONS = {"not","no","never","neither","nor","nothing","nobody","nowhere",
             "cannot","won","don","doesn","isn","wasn","weren","haven","hadn",
             "couldn","wouldn","shouldn"}

STOPWORDS -= NEGATIONS

CONTRACTIONS = {
    "won't":"will not","can't":"cannot","n't":" not","'re":" are","'s":" is",
    "'d":" would","'ll":" will","'ve":" have","'m":" am","it's":"it is",
    "that's":"that is","there's":"there is","they're":"they are",
    "we're":"we are","i'm":"i am","i've":"i have","i'll":"i will","i'd":"i would",
    "you're":"you are","he's":"he is","she's":"she is","let's":"let us"
}

LEMMA_RULES = [
    (r"nesses$","ness"),(r"ness$",""),(r"ments$","ment"),(r"ment$",""),
    (r"ations$","ate"),(r"ation$","ate"),(r"ities$","ity"),(r"ings$",""),
    (r"ing$",""),(r"edly$",""),(r"ly$",""),(r"ied$","y"),(r"ies$","y"),
    (r"ed$",""),(r"er$",""),(r"est$",""),(r"s$",""),
]

def lemmatize(w):
    if len(w) <= 3: return w
    for pat, rep in LEMMA_RULES:
        n = re.sub(pat, rep, w)
        if n != w and len(n) >= 2: return n
    return w

def preprocess(text):
    """
    NLP Pipeline:
      1. Lowercase  2. Expand contractions  3. Remove special chars
      4. Tokenise   5. Remove stopwords (preserve negations)  6. Lemmatise
    """
    text = text.lower()
    for k, v in CONTRACTIONS.items(): text = text.replace(k, v)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS]
    tokens = [lemmatize(t) for t in tokens if len(t) > 1]
    return " ".join(tokens)

## LOADING DATASET

In [6]:
df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")

df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df = df.dropna(subset=["Review Text"])

df = df.rename(columns={
    "Clothing ID": "product_id",
    "Review Text": "review_text",
    "Rating": "rating"
})

# Combine product info
df["product_name"] = (
    df["Division Name"].fillna("") + " - " +
    df["Department Name"].fillna("") + " - " +
    df["Class Name"].fillna("")
)

# Combine title + review (better NLP signal)
df["text_combined"] = df["Title"].fillna("") + " " + df["review_text"]

# Sentiment labeling
df["sentiment"] = df["rating"].apply(
    lambda r: "positive" if r >= 4 else ("neutral" if r == 3 else "negative")
)

# NLP preprocessing
df["clean_text"] = df["text_combined"].apply(preprocess)

print("="*65)
print(" WOMEN'S CLOTHING REVIEW SENTIMENT ANALYZER")
print("="*65)
print(f"\n Reviews: {len(df)} | Unique Products: {df['product_id'].nunique()}")

 WOMEN'S CLOTHING REVIEW SENTIMENT ANALYZER

 Reviews: 22641 | Unique Products: 1179


## TF - IDF

In [7]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=3000,
    sublinear_tf=True
)

X = tfidf.fit_transform(df["clean_text"])
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## MODEL DEVELOPMENT AND EVALUATION

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVC": LinearSVC(class_weight="balanced")
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")

    results[name] = (model, acc, f1)

    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(classification_report(y_test, preds, target_names=le.classes_))

# Best model
best_name = max(results, key=lambda k: results[k][2])
best_model = results[best_name][0]

print(f"\nBEST MODEL: {best_name}")


Logistic Regression
Accuracy: 0.7903
F1 Score: 0.8090
              precision    recall  f1-score   support

    negative       0.53      0.65      0.58       593
     neutral       0.34      0.50      0.41       706
    positive       0.96      0.86      0.91      4362

    accuracy                           0.79      5661
   macro avg       0.61      0.67      0.63      5661
weighted avg       0.84      0.79      0.81      5661


Naive Bayes
Accuracy: 0.8066
F1 Score: 0.7608
              precision    recall  f1-score   support

    negative       0.71      0.27      0.39       593
     neutral       0.42      0.14      0.21       706
    positive       0.83      0.99      0.90      4362

    accuracy                           0.81      5661
   macro avg       0.65      0.47      0.50      5661
weighted avg       0.76      0.81      0.76      5661


Linear SVC
Accuracy: 0.8110
F1 Score: 0.8152
              precision    recall  f1-score   support

    negative       0.54      0.59  

## KEYWORDS BY SENTIMENT

In [9]:
feature_names = np.array(tfidf.get_feature_names_out())

def get_keywords(model, classes, top_n=10):
    keywords = {}
    for i, cls in enumerate(classes):
        if hasattr(model, "coef_"):
            coefs = model.coef_[i] if len(model.coef_) > 1 else model.coef_[0]
            top = np.argsort(coefs)[-top_n:]
        else:
            coefs = model.feature_log_prob_[i]
            top = np.argsort(coefs)[-top_n:]

        keywords[cls] = feature_names[top].tolist()
    return keywords

keywords = get_keywords(best_model, le.classes_)

print("\nKEYWORDS BY SENTIMENT")
for k, v in keywords.items():
    print(f"{k}: {v[:5]}")


KEYWORDS BY SENTIMENT
negative: ['acros bust', 'ill', 'order regular', 'not', 'want like']
neutral: ['color bright', 'quality love', 'meh', 'material little', 'receiv dres']
positive: ['comfy', 'pleas', 'unique', 'happy', 'comfortable']


## TOP PRODUCTS

In [10]:
top_products = df["product_id"].value_counts().head(10).index

print("\nTOP PRODUCT INSIGHTS")
for pid, grp in df[df["product_id"].isin(top_products)].groupby("product_id"):
    avg_rating = grp["rating"].mean()
    sentiment_counts = grp["sentiment"].value_counts(normalize=True) * 100

    print(f"\nProduct {pid}")
    print(f"Avg Rating: {avg_rating:.2f}")
    print(sentiment_counts)


TOP PRODUCT INSIGHTS

Product 829
Avg Rating: 4.19
sentiment
positive    76.953125
neutral     13.085938
negative     9.960938
Name: proportion, dtype: float64

Product 862
Avg Rating: 4.19
sentiment
positive    77.506427
neutral     11.825193
negative    10.668380
Name: proportion, dtype: float64

Product 868
Avg Rating: 3.93
sentiment
positive    67.874396
neutral     17.149758
negative    14.975845
Name: proportion, dtype: float64

Product 872
Avg Rating: 4.37
sentiment
positive    83.622351
neutral     10.597303
negative     5.780347
Name: proportion, dtype: float64

Product 895
Avg Rating: 4.27
sentiment
positive    79.166667
neutral     11.458333
negative     9.375000
Name: proportion, dtype: float64

Product 936
Avg Rating: 4.31
sentiment
positive    79.022989
neutral     14.080460
negative     6.896552
Name: proportion, dtype: float64

Product 1078
Avg Rating: 4.19
sentiment
positive    77.102330
neutral     13.677812
negative     9.219858
Name: proportion, dtype: float64

Pro

## SAVING RESULTS

In [11]:
output = {
    "best_model": best_name,
    "keywords": keywords,
}

with open("results.json", "w") as f:
    json.dump(output, f, indent=2)

print("\nResults saved to results.json")
print("="*65)


Results saved to results.json


## END